In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
import random

import os
import pickle
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch
import copy

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function

from sklearn.preprocessing import StandardScaler

from functions import CVaRSolver, SPOPlus, VARasNN
import functions

In [2]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data.csv")

In [3]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')

## Experimental Pipeline

For each of 5 randomly sampled 15-asset subsets of the S&P 500:

For each test month (rolling walk-forward evaluation over the last 12 months of the sample):

1. **Data preparation**: Split data into training, validation (last 6 months before test month), and test (single month) sets. Features $z_{t-1}$ consist of the last 3 months of returns (VAR(3) lag structure).

2. **Scenario matrix**: Bootstrap 1000 return scenarios from the training period to approximate the empirical return distribution for the CVaR constraint. A separate scenario matrix including the validation period is used for test-month inference.

3. **Oracle solutions**: Precompute the optimal portfolio $w^\star(y_t)$ for every training and validation instance by solving the CVaR-LP with true returns as input. These are required to compute regret during training and early stopping.

4. **Loss normalization**: Compute SPO+ and MSE scale factors on the initialized model to ensure the two loss components are comparable when combined.

5. **Model training**: For each $\gamma \in \{0, 0.25, 0.5, 0.75, 1.0\}$, train a VAR(3) model (implemented as a linear layer) by minimizing the combined loss
$$\mathcal{L}_{\text{combined}} = \gamma \cdot \mathcal{L}_{\text{SPO+}} + (1 - \gamma) \cdot \mathcal{L}_{\text{MSE}}$$
using Adam with learning rate $10^{-3}$, batch size 16, and early stopping based on validation regret with patience 2. Model weights are restored from the epoch achieving the lowest validation regret.

6. **Test inference**: The trained model predicts returns $\hat{y}_{\text{test}}$ for the test month. The CVaR-LP is solved with $\hat{y}_{\text{test}}$ to obtain portfolio weights $\hat{w}$, and again with true returns $y_{\text{test}}$ to obtain the oracle portfolio $w^\star$. Realized return, oracle return, and regret are recorded.

7. **Checkpointing**: Results are saved to disk after every (subset, test month) combination to prevent data loss.

Results are aggregated across all test months and asset subsets to compare DFL ($\gamma > 0$) against pure MSE training ($\gamma = 0$) in terms of realized portfolio return, test regret, and prediction accuracy.

## Specify parameters

In [4]:
beta = 0.09                 # CVaR threshold (justification in thesis)

n_test_months = 12          # last year as test period

gamma_levels = [0, 0.25, 0.5, 0.75, 1.0]    # similiar to Lee et al.

n_epochs = 10               # HYPERPARAM: number of epochs
val_length = 6              # HYPERPARAM: validation length
max_lag = 3                 # HYPERPARAM: lags of VAR model

## Final Loop

In [5]:
results = []

for data_seed in range(1,6):
    # prepare random return data subset
    random.seed(data_seed)
    cols_subset = random.sample(list(return_matrix.columns), 15)
    return_matrix_subset = return_matrix[cols_subset]
    # Save preprocessed return data subset
    return_matrix_subset.to_csv(f"../data/processed/return_matrix_random_seed{data_seed}.csv", index=False)
    
    X, Y = functions.create_time_series_data_with_lags(return_matrix_subset, max_lag)

    
    for test_index in tqdm(range(1,n_test_months+1), desc="test months"):

        print(f"\nDataset {data_seed}; Test Index = {test_index}\n")
        print("Preparing Data...")

        # data preparation
        X_train, X_val, X_test, Y_train, Y_val, Y_test = functions.create_train_test_split(X, Y, test_index, val_length)
        train_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index + val_length, num_scenarios=1000, random_seed=42)
        S, N = train_scenario_loss_matrix.shape # S scenarios × N assets

        # solver
        train_solver = CVaRSolver(
            loss_matrix=train_scenario_loss_matrix,
            N=N,
            S=S,
            alpha=0.95, # CVaR confidence level
            beta=beta
        )

        # Precompute oracle solutions
        oracle_solutions = [
            train_solver.solve(c = - mu).copy()
            for mu in tqdm(Y_train, desc="Computing oracle solutions")
        ]
        oracle_tensor = torch.tensor(
            np.array(oracle_solutions),
            dtype=torch.float32
        )

        # Precompute oracle solutions for validation set
        oracle_val_solutions = [
            train_solver.solve(c=-mu).copy()
            for mu in Y_val
        ]
        oracle_val_tensor = torch.tensor(
            np.array(oracle_val_solutions),
            dtype=torch.float32
        )


        # Prepare dataset for pytorch training
        x_scaler = StandardScaler()
        X_train = x_scaler.fit_transform(X_train)
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
        train_dataset = TensorDataset(
            X_train_tensor,
            Y_train_tensor,
            oracle_tensor
        )
        batch_size = 16                                                          # HYPERPARAM: Batch size
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,        # TensorDataset shuffles all tensors together
            drop_last=False
        )

        # scale validation data using the training scaler
        X_val_scaled = x_scaler.transform(X_val)
        X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
        Y_val_tensor = torch.tensor(Y_val, dtype=torch.float32)

        # specify model for scale computation
        model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
        criterion = nn.MSELoss()
        # compute scale factor for the two losses to make them comparable
        spo_scale, mse_scale = functions.compute_loss_normalization(model, train_loader, train_solver, criterion)
        print(f"SPO+ scale: {spo_scale:.6f}", f"; MSE  scale: {mse_scale:.6f}\n")

        print("Train Models for different loss combinations:")
        
        for gamma in gamma_levels:
            
            print(f"\nGamma = {gamma} (weight of SPO+ loss)\n")

            # specify model and optimizer
            torch.manual_seed(42)
            model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=1e-3                                                             # HYPERPARAM: Learning rate
            )
            criterion = nn.MSELoss()

            # Train model and retrieve training history
            history = functions.train_model(model,
                                            n_epochs,
                                            train_loader,
                                            optimizer,
                                            criterion,
                                            train_solver,
                                            spo_scale,
                                            mse_scale,
                                            gamma,
                                            X_val_tensor,
                                            Y_val_tensor,
                                            oracle_val_tensor,
                                            early_stopping_patience=2
                                            )

            # Inference
            
            # Scale test data using training scaler
            X_test_scaled = x_scaler.transform(X_test.reshape(1, -1))

            X_test_tensor = torch.tensor(
                X_test_scaled,
                dtype=torch.float32
            )

            # Predict expected returns
            model.eval()
            with torch.no_grad():
                mu_hat_test = model(X_test_tensor)

            mu_hat_test = mu_hat_test.cpu().numpy()[0]

            test_mse = np.mean((mu_hat_test - Y_test)**2)

            # Compute portfolio weights from predicted returns with test solver
            test_scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix_subset, test_index, num_scenarios=1000, random_seed=42)
            # test solver
            test_solver = CVaRSolver(
                loss_matrix=test_scenario_loss_matrix,
                N=N,
                S=S,
                alpha=0.95, # CVaR confidence level
                beta=beta
            )

            # Compute portfolio weights using test solver (train + val scenario matrix)
            w_hat = test_solver.solve(c=-mu_hat_test).copy()
            w_oracle = test_solver.solve(c=-Y_test).copy()

            # Realized returns
            realized_return = float(Y_test @ w_hat)
            oracle_return   = float(Y_test @ w_oracle)

            # True regret: how much return we lost by using predicted rather than true returns
            test_regret = oracle_return - realized_return

            results.append({

                # id
                "run_id": f"seed{data_seed}_g{gamma}_t{test_index}_b{beta}",

                # experiment settings
                "data_seed": data_seed,
                "beta": beta,
                "gamma": gamma,
                "test_index": test_index,

                # dimensions
                "n_train": len(X_train),
                "n_assets": N,
                "n_scenarios": S,

                # normalization factors
                "spo_scale": spo_scale,
                "mse_scale": mse_scale,

                # forecasting results
                "Y_hat_test": mu_hat_test,
                "Y_test": Y_test,
                "test_mse": test_mse,

                # portfolio results
                "weights": w_hat,
                "w_oracle": w_oracle,
                "realized_return": realized_return,
                "oracle_return": oracle_return,
                "test_regret": test_regret,

                # training history
                "history": history,
                "final_spo_loss": history["spo_loss"][-1],
                "final_mse_loss": history["mse_loss"][-1],
                "final_combined_loss": history["combined_loss"][-1],
                "early_stopping_epoch": history["early_stopping_epoch"],
                "best_epoch": history["best_epoch"],
                "best_val_regret": min(history["val_regret"])
            })

        print("\n---------------------------------------------------")

        # save results after every combination of data_seed and test_index
        with open("results_checkpoint_tmp.pkl", "wb") as f:
            pickle.dump(results, f)
        os.replace("results_checkpoint_tmp.pkl", "results_checkpoint.pkl")

test months:   0%|          | 0/12 [00:00<?, ?it/s]


Dataset 1; Test Index = 1

Preparing Data...


Computing loss scales for normalization: 100%|██████████| 18/18 [00:28<00:00,  1.57s/it]


SPO+ scale: 0.998629 ; MSE  scale: 0.353811

Train Models for different loss combinations:

Gamma = 0 (weight of SPO+ loss)

Epoch 1: combined=0.884770 | SPO+ scaled=0.916393 | MSE scaled=0.884770 | val_regret=0.034198  ✓ regret improved
Epoch 2: combined=0.652348 | SPO+ scaled=0.799518 | MSE scaled=0.652348 | val_regret=0.031970  ✓ regret improved
Epoch 3: combined=0.508180 | SPO+ scaled=0.700029 | MSE scaled=0.508180 | val_regret=0.034602
Epoch 4: combined=0.410735 | SPO+ scaled=0.636750 | MSE scaled=0.410735 | val_regret=0.036639
Early stopping at epoch 4. Best val regret: 0.031970

Gamma = 0.25 (weight of SPO+ loss)

Epoch 1: combined=0.892924 | SPO+ scaled=0.915010 | MSE scaled=0.885562 | val_regret=0.034236  ✓ regret improved
Epoch 2: combined=0.687782 | SPO+ scaled=0.790922 | MSE scaled=0.653402 | val_regret=0.032005  ✓ regret improved
Epoch 3: combined=0.554642 | SPO+ scaled=0.688667 | MSE scaled=0.509967 | val_regret=0.035543
Epoch 4: combined=0.465939 | SPO+ scaled=0.622954 |

test months:   8%|▊         | 1/12 [10:44<1:58:12, 644.73s/it]

...


Dataset 5; Test Index = 12

Preparing Data...


Computing loss scales for normalization: 100%|██████████| 17/17 [00:18<00:00,  1.10s/it]


SPO+ scale: 0.869114 ; MSE  scale: 0.328623

Train Models for different loss combinations:

Gamma = 0 (weight of SPO+ loss)

Epoch 1: combined=0.916218 | SPO+ scaled=0.941751 | MSE scaled=0.916218 | val_regret=0.042366  ✓ regret improved
Epoch 2: combined=0.695274 | SPO+ scaled=0.824688 | MSE scaled=0.695274 | val_regret=0.042202  ✓ regret improved
Epoch 3: combined=0.540038 | SPO+ scaled=0.731307 | MSE scaled=0.540038 | val_regret=0.039918  ✓ regret improved
Epoch 4: combined=0.428576 | SPO+ scaled=0.655087 | MSE scaled=0.428576 | val_regret=0.039149  ✓ regret improved
Epoch 5: combined=0.338941 | SPO+ scaled=0.584511 | MSE scaled=0.338941 | val_regret=0.039043  ✓ regret improved
Epoch 6: combined=0.267307 | SPO+ scaled=0.519649 | MSE scaled=0.267307 | val_regret=0.038712  ✓ regret improved
Epoch 7: combined=0.213350 | SPO+ scaled=0.463826 | MSE scaled=0.213350 | val_regret=0.038776
Epoch 8: combined=0.169001 | SPO+ scaled=0.414538 | MSE scaled=0.169001 | val_regret=0.039668
Early sto

test months: 100%|██████████| 12/12 [1:52:37<00:00, 563.16s/it]

Epoch 5: combined=0.611095 | SPO+ scaled=0.611095 | MSE scaled=0.596158 | val_regret=0.040115
Early stopping at epoch 5. Best val regret: 0.038135

---------------------------------------------------


In [7]:
len(results)

300

In [8]:
with open("results_checkpoint.pkl", "rb") as f:
    results = pickle.load(f)

In [ ]:
df = pd.DataFrame([{
    'data_seed': r['data_seed'],
    'gamma': r['gamma'],
    'test_index': r['test_index'],
    'test_regret': r['test_regret'],
    'realized_return': r['realized_return'],
    'best_epoch': r['best_epoch'],
} for r in results])

print(df.groupby('gamma')['test_regret'].agg(['mean', 'std', 'min', 'max']))
print(f"\nNegative regrets: {(df['test_regret'] < 0).sum()}")
print(f"\nBest epoch distribution:\n{df.groupby('gamma')['best_epoch'].mean()}")

           mean       std       min       max
gamma                                        
0.00   0.047936  0.023897  0.006005  0.128023
0.25   0.047270  0.023745  0.006005  0.131792
0.50   0.047923  0.023854  0.006005  0.131792
0.75   0.046803  0.022637  0.006005  0.128554
1.00   0.046526  0.023472  0.005596  0.128554

Negative regrets: 0

Best epoch distribution:
gamma
0.00    2.254237
0.25    2.224138
0.50    2.483333
0.75    2.321429
1.00    2.053571
Name: best_epoch, dtype: float64
